# 📚 The Unofficial Campus Survival Guide
**AI201 Project 1 — RAG System**

A Retrieval-Augmented Generation (RAG) system that makes student-generated campus survival knowledge searchable and answerable.

**Domain**: General college survival — registration, financial aid, mental health, campus resources  
**Sources**: Reddit threads (r/college, r/financialaid, r/StudentLoans) and student-written guides  
**Stack**: sentence-transformers (all-MiniLM-L6-v2) · ChromaDB · Groq (llama-3.3-70b-versatile)

---
## How to use this notebook
Run cells **top to bottom** on first launch:
1. **Setup** — install deps, load env
2. **Milestone 3** — build document pipeline
3. **Milestone 4** — embed + test retrieval
4. **Milestone 5** — generate grounded answers, use query interface
5. **Milestone 6** — evaluation report

After setup, jump straight to **Section 5 (Query Interface)** for interactive use.

## 🔧 Section 1: Setup

In [10]:
# Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'chromadb', 'groq', 'python-dotenv',
    'ipywidgets'])
print('✅ Dependencies installed')

✅ Dependencies installed


In [11]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads GROQ_API_KEY from .env

if os.environ.get('GROQ_API_KEY'):
    print('✅ GROQ_API_KEY loaded')
else:
    print('⚠️  GROQ_API_KEY not found — copy .env.example to .env and add your key')
    print('   Get a free key at https://console.groq.com')

✅ GROQ_API_KEY loaded


## 📄 Section 2: Milestone 3 — Document Pipeline

In [12]:
from ingest import run_pipeline, inspect_chunks

# Run the full ingestion pipeline
chunks = run_pipeline(docs_dir='docs')
print(f'\n✅ Pipeline complete: {len(chunks)} total chunks')

[ingest] Loaded 12 documents from 'docs'
[ingest] Total chunks produced: 64

✅ Pipeline complete: 64 total chunks


In [13]:
# Milestone 3 Checkpoint: inspect 5 random chunks
# Each should be readable, substantive, and self-contained
inspect_chunks(chunks, n=5)


CHUNK INSPECTION — 5 random samples

--- Chunk 1 | Source: 01_financial_aid_appeal.txt | Index: 0 ---
So my financial aid was cut by $4,000 this year and I was completely blindsided. Here's what worked for me when I appealed:

First, write a formal appeal letter. Be specific — don't just say "I need more money." Explain exactly what changed in your financial situation. I listed my dad's job loss, the medical bills from my mom's surgery, and attached documentation for all of it. The financial aid office wants to see receipts, termination letters, anything concrete.
[468 chars]

--- Chunk 2 | Source: 07_work_study_parttime.txt | Index: 0 ---
Balancing a part-time job with college — what actually works:

Cap your work hours at 15-20 per week during the semester. Research consistently shows academic performance drops significantly above 20 hours. If you're working more than 20 hours, have a real conversation with yourself about whether you need to adjust your course load, look for higher-

In [14]:
# Chunk stats
sizes = [len(c['text']) for c in chunks]
sources = list(set(c['source'] for c in chunks))

print(f'Documents loaded:   {len(sources)}')
print(f'Total chunks:       {len(chunks)}')
print(f'Avg chunk size:     {sum(sizes)//len(sizes)} chars')
print(f'Min chunk size:     {min(sizes)} chars')
print(f'Max chunk size:     {max(sizes)} chars')
print(f'\nChunks per document:')
from collections import Counter
for src, count in sorted(Counter(c['source'] for c in chunks).items()):
    print(f'  {src}: {count} chunks')

Documents loaded:   12
Total chunks:       64
Avg chunk size:     368 chars
Min chunk size:     250 chars
Max chunk size:     497 chars

Chunks per document:
  01_financial_aid_appeal.txt: 5 chunks
  02_fafsa_tips.txt: 6 chunks
  03_registration_waitlists.txt: 6 chunks
  04_mental_health_resources.txt: 6 chunks
  05_academic_probation.txt: 5 chunks
  06_student_loans.txt: 5 chunks
  07_work_study_parttime.txt: 5 chunks
  08_campus_food_emergency_funds.txt: 6 chunks
  09_first_gen_tips.txt: 6 chunks
  10_registration_strategies.txt: 4 chunks
  11_grade_appeals.txt: 5 chunks
  12_campus_health_services.txt: 5 chunks


## 🔍 Section 3: Milestone 4 — Embed and Test Retrieval

In [ ]:
from embed import build_vector_store

# Build vector store (takes ~30s on first run, instant after)
# Set reset=True to rebuild from scratch
collection = build_vector_store(chunks, reset=False)
print(f'\n✅ Vector store ready: {collection.count()} chunks indexed')

[embed] Loading all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Milestone 4 Checkpoint: test retrieval on 3 evaluation questions
# Good retrieval: on-topic chunks, distance scores < 0.5

from query import retrieve

test_queries = [
    'What should I do if my financial aid was cut?',
    'What campus mental health resources are available?',
    'How do I get off a course waitlist?',
]

for q in test_queries:
    print(f'\n{"="*60}')
    print(f'Query: {q}')
    print('='*60)
    results = retrieve(q, k=4)
    for i, r in enumerate(results, 1):
        print(f'\n  [{i}] Source: {r["source"]} | Distance: {r["distance"]}')
        print(f'      {r["text"][:200]}...')

## 🤖 Section 4: Milestone 5 — Grounded Generation

In [ ]:
# Test end-to-end generation on 2 queries
from query import ask

test_q = 'What should I do if my financial aid was reduced or cut?'
print(f'Question: {test_q}\n')

result = ask(test_q, verbose=True)
print(f'\n{"-"*60}')
print('ANSWER:')
print(result['answer'])
print(f'\nSOURCES: {result["sources"]}')

In [ ]:
# Grounding test: ask something NOT in the documents
# The system should decline rather than hallucinate
out_of_scope = 'What is the best programming language for machine learning?'
print(f'Out-of-scope question: {out_of_scope}\n')

oos_result = ask(out_of_scope)
print('ANSWER:')
print(oos_result['answer'])

## 💬 Section 5: Query Interface
**Interactive widget — enter any campus survival question below.**  
The system retrieves relevant chunks and generates a grounded, cited answer.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from query import ask

# UI components
question_input = widgets.Textarea(
    placeholder='Ask a campus survival question...\ne.g. "What should I do if I missed the FAFSA deadline?"',
    layout=widgets.Layout(width='100%', height='80px')
)

submit_btn = widgets.Button(
    description='Ask the Guide',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='160px', height='36px')
)

output_area = widgets.Output()

# Example question buttons
examples = [
    'What should I do if my financial aid was reduced?',
    'How do I get off a waitlist?',
    'What mental health resources are available on campus?',
    'I missed the FAFSA deadline — what are my options?',
    'How do I balance a part-time job with classes?',
]

def make_example_btn(q):
    btn = widgets.Button(
        description=q[:50] + ('...' if len(q) > 50 else ''),
        button_style='',
        layout=widgets.Layout(width='100%', height='32px')
    )
    def on_click(b):
        question_input.value = q
    btn.on_click(on_click)
    return btn

example_btns = [make_example_btn(q) for q in examples]

def on_submit(btn):
    question = question_input.value.strip()
    if not question:
        return
    with output_area:
        clear_output()
        display(HTML('<p><em>🔍 Searching documents and generating answer...</em></p>'))
    
    result = ask(question)
    
    sources_html = ''.join(f'<li><code>{s}</code></li>' for s in result['sources'])
    chunks_html = ''.join(
        f'<details><summary>Chunk {i+1}: {c["source"]} (distance: {c["distance"]})</summary>'
        f'<pre style="white-space:pre-wrap;font-size:12px">{c["text"]}</pre></details>'
        for i, c in enumerate(result['chunks'])
    )
    
    answer_html = result['answer'].replace('\n', '<br>')
    
    html = f'''
    <div style="border:1px solid #ddd;border-radius:8px;padding:16px;margin-top:8px">
      <h3 style="margin-top:0">📖 Answer</h3>
      <p style="line-height:1.7">{answer_html}</p>
      <hr>
      <h4>📎 Retrieved from:</h4>
      <ul>{sources_html}</ul>
      <hr>
      <h4>🔍 Retrieved chunks (expand to inspect)</h4>
      {chunks_html}
    </div>
    '''
    
    with output_area:
        clear_output()
        display(HTML(html))

submit_btn.on_click(on_submit)

# Layout
display(HTML('<h3>🎓 Ask the Unofficial Campus Survival Guide</h3>'))
display(HTML('<p style="color:#666">Questions answered using only student-generated knowledge from Reddit and campus forums.</p>'))
display(HTML('<strong>Try an example:</strong>'))
display(widgets.VBox(example_btns))
display(HTML('<br><strong>Or type your own question:</strong>'))
display(question_input)
display(submit_btn)
display(output_area)

---
## 📊 Section 6: Milestone 6 — Evaluation Report

5 test questions with ground-truth answers, system responses, and accuracy judgments.

| # | Question | Expected answer | System accuracy |
|---|----------|----------------|-----------------|
| 1 | What should I do if my financial aid was reduced? | Appeal in writing, provide documentation, request professional judgment review, ask about emergency funds | *run cell below* |
| 2 | How do I get off a course waitlist? | Email professor directly, show up first day, ask about department override, watch for drops | *run cell below* |
| 3 | What campus mental health resources exist? | CAPS (Counseling and Psychological Services), 988 lifeline, Crisis Text Line, peer counseling, wellness center | *run cell below* |
| 4 | I missed the FAFSA deadline — what are my options? | File anyway, contact financial aid office, check institutional deadlines separately, look for emergency funds | *run cell below* |
| 5 | How do I balance a part-time job with full-time classes? | Cap at 15-20 hrs/week, prioritize work-study, use time blocks, use campus resources to reduce other costs | *run cell below* |

In [7]:
# Run all 5 evaluation questions and display results
from query import ask
from IPython.display import display, HTML

eval_questions = [
    {
        'q': 'What should I do if my financial aid was reduced or cut?',
        'expected': 'Appeal in writing with documentation, request professional judgment review, ask about emergency funds, contact financial aid office directly'
    },
    {
        'q': 'How do I get off a course waitlist?',
        'expected': 'Email professor directly, show up first day, ask about department override, watch enrollment numbers during add/drop'
    },
    {
        'q': 'What campus mental health resources exist for students?',
        'expected': 'CAPS/counseling center (often free), 988 crisis line, Crisis Text Line (text HOME to 741741), peer counseling, student wellness center'
    },
    {
        'q': 'I missed the FAFSA deadline — what are my options?',
        'expected': 'File anyway, contact financial aid office, check institutional priority deadlines separately, look for emergency student funds'
    },
    {
        'q': 'How do I balance a part-time job with full-time classes?',
        'expected': 'Cap work at 15-20 hrs/week, prioritize work-study jobs, use time block scheduling, use campus food pantry and emergency funds to reduce financial pressure'
    },
]

print('Running evaluation... this may take 30-60 seconds\n')
results = []
for item in eval_questions:
    r = ask(item['q'])
    results.append({'question': item['q'], 'expected': item['expected'],
                    'answer': r['answer'], 'sources': r['sources'],
                    'chunks': r['chunks']})
    print(f'  ✓ {item["q"][:60]}...')

print('\nEvaluation complete. Scroll down to see results.')

Running evaluation... this may take 30-60 seconds



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  ✓ What should I do if my financial aid was reduced or cut?...
  ✓ How do I get off a course waitlist?...
  ✓ What campus mental health resources exist for students?...
  ✓ I missed the FAFSA deadline — what are my options?...
  ✓ How do I balance a part-time job with full-time classes?...

Evaluation complete. Scroll down to see results.


In [ ]:
# Display evaluation report
# After reviewing, manually fill in the accuracy judgment for each question
# Judgment options: Accurate | Partially accurate | Inaccurate

for i, r in enumerate(results, 1):
    print(f'{'='*70}')
    print(f'QUESTION {i}: {r["question"]}')
    print(f'{'='*70}')
    print(f'\nEXPECTED ANSWER:')
    print(f'  {r["expected"]}')
    print(f'\nSYSTEM RESPONSE:')
    print(r['answer'])
    print(f'\nSOURCES CITED: {r["sources"]}')
    print(f'\nCHUNKS RETRIEVED:')
    for j, c in enumerate(r['chunks'], 1):
        print(f'  [{j}] {c["source"]} (distance: {c["distance"]})')
    print(f'\nACCURACY JUDGMENT: [ fill in after review — Accurate / Partially accurate / Inaccurate ]')
    print()

### Failure Case Analysis

*(Fill this in after running the evaluation above)*

**Question that failed or partially failed:**  
Write the question here.

**What the system returned:**  
Summarize the system's answer.

**Why it failed — specific pipeline cause:**  
Example: *"The relevant information about [X] was split across a chunk boundary in [filename]. The retrieval returned the first half of the advice (chunk 2) but not the second half (chunk 3), because the embedding for chunk 2 alone didn't match the query semantics strongly enough. Distance score was 0.48 — borderline. Increasing chunk overlap from 75 to 150 characters would likely fix this."*

---
### Spec Reflection

**One way the spec helped:** The chunking strategy section forced me to think about chunk size *before* seeing any retrieval results, which meant my initial parameters were much closer to right than if I'd guessed. The 500-char/75-overlap decision was grounded in actual document structure (short Reddit comments), not arbitrary.

**One way implementation diverged from the spec:** *(Fill in after building — e.g., "I planned to use paragraph splitting exclusively but found some documents had no clear paragraph breaks, so I added a fallback hard-character split.")*

---
### AI Usage Log

1. **Generating `ingest.py`**: Provided the Chunking Strategy section from planning.md (500 chars, 75 overlap, recursive paragraph-first splitting). Claude generated the `chunk_text()` function. Reviewed and changed: replaced the initial `langchain` dependency with a pure-Python recursive implementation to reduce dependencies.

2. **Generating `query.py` system prompt**: Provided the grounding requirement and source attribution requirement. Claude generated a system prompt. Tightened the grounding instruction from "try to answer from documents" to "ONLY use the provided documents" and added the explicit refusal format for out-of-scope questions.